<a href="https://colab.research.google.com/github/Aman3756/seattle-weather-analysis/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()   # choose DailyDelhiClimateTest.csv
data = pd.read_csv("DailyDelhiClimateTest.csv")
data.head()

Saving DailyDelhiClimateTest.csv to DailyDelhiClimateTest (2).csv


,date,meantemp,humidity,wind_speed,meanpressure
0,2017-01-01,15.913043,85.869565,2.743478,59.000000
1,2017-01-02,18.500000,77.222222,2.894444,1018.277778
2,2017-01-03,17.111111,81.888889,4.016667,1018.333333
3,2017-01-04,18.700000,70.050000,4.545000,1015.700000
4,2017-01-05,18.388889,74.944444,3.300000,1014.333333


In [6]:
print("Rows and columns:", data.shape)
data.info()
print(data.isnull().sum())
data.describe()
print("First date:", data['date'].min())
print("Last date:", data['date'].max())

Rows and columns: (114, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          114 non-null    object 
 1   meantemp      114 non-null    float64
 2   humidity      114 non-null    float64
 3   wind_speed    114 non-null    float64
 4   meanpressure  114 non-null    float64
dtypes: float64(4), object(1)
memory usage: 4.6+ KB
date            0
meantemp        0
humidity        0
wind_speed      0
meanpressure    0
dtype: int64
First date: 2017-01-01
Last date: 2017-04-24


In [7]:
data['date'] = pd.to_datetime(data['date'])
data = data.rename(columns={'meantemp': 'temp',
                            'meanpressure': 'pressure'})

data['month'] = data['date'].dt.month
data['month_name'] = data['date'].dt.month_name()
data['week'] = (data['date'].dt.dayofyear - 1) // 7 + 1
data['temp_f'] = data['temp'] * 9 / 5 + 32

data['temp_category'] = np.where(data['temp'] > 25, 'Warm',
                         np.where(data['temp'] > 15, 'Mild', 'Cold'))
data['humid_level'] = np.where(data['humidity'] > 60, 'High', 'Low')
data.head()

,date,temp,humidity,wind_speed,pressure,month,month_name,week,temp_f,temp_category,humid_level
0,2017-01-01,15.913043,85.869565,2.743478,59.000000,1,January,1,60.643478,Mild,High
1,2017-01-02,18.500000,77.222222,2.894444,1018.277778,1,January,1,65.300000,Mild,High
2,2017-01-03,17.111111,81.888889,4.016667,1018.333333,1,January,1,62.800000,Mild,High
3,2017-01-04,18.700000,70.050000,4.545000,1015.700000,1,January,1,65.660000,Mild,High
4,2017-01-05,18.388889,74.944444,3.300000,1014.333333,1,January,1,65.100000,Mild,High


In [8]:
monthly = data.groupby('month_name')[['temp', 'humidity', 'wind_speed']].mean().round(1)
print("Average values by month:")
print(monthly)

month_temp = data.groupby('month')['temp'].mean()
print("Warmest month:", month_temp.idxmax())
print("Coldest month:", month_temp.idxmin())

Average values by month:
            temp  humidity  wind_speed
month_name                            
April       30.8      30.4         9.1
February    18.3      64.4         7.6
January     15.7      77.2         7.2
March       23.8      48.0         8.9
Warmest month: 4
Coldest month: 1


In [9]:
weekly = data.groupby('week')['temp'].mean().round(1)
print("Average temperature by week number:")
print(weekly)
print("Change from first week to last week:", round(weekly.iloc[-1] - weekly.iloc[0], 1), "°C")

Average temperature by week number:
week
1     17.5
2     13.1
3     14.4
4     17.6
5     16.4
6     16.0
7     18.5
8     20.5
9     21.4
10    20.5
11    20.3
12    26.2
13    30.1
14    29.2
15    29.3
16    33.2
17    32.4
Name: temp, dtype: float64
Change from first week to last week: 14.9 °C


In [10]:
print("Days in each temperature category:")
print(data['temp_category'].value_counts())
print("Percentage share:")
print((data['temp_category'].value_counts() / len(data) * 100).round(1))
print("Category count by month:")
print(pd.crosstab(data['month_name'], data['temp_category']))

Days in each temperature category:
temp_category
Mild    67
Warm    34
Cold    13
Name: count, dtype: int64
Percentage share:
temp_category
Mild    58.8
Warm    29.8
Cold    11.4
Name: count, dtype: float64
Category count by month:
temp_category  Cold  Mild  Warm
month_name                     
April             0     0    24
February          1    27     0
January          12    19     0
March             0    21    10


In [11]:
print("Average humidity for each temperature category:")
print(data.groupby('temp_category')['humidity'].mean().round(1))
print("Average wind speed for each temperature category:")
print(data.groupby('temp_category')['wind_speed'].mean().round(1))
print("Average temperature on high vs low humidity days:")
print(data.groupby('humid_level')['temp'].mean().round(1))

Average humidity for each temperature category:
temp_category
Cold    77.5
Mild    63.9
Warm    33.1
Name: humidity, dtype: float64
Average wind speed for each temperature category:
temp_category
Cold    8.3
Mild    7.4
Warm    9.5
Name: wind_speed, dtype: float64
Average temperature on high vs low humidity days:
humid_level
High    16.7
Low     26.2
Name: temp, dtype: float64


In [12]:
print(data[['temp', 'humidity', 'wind_speed', 'pressure']].corr().round(2))

            temp  humidity  wind_speed  pressure
temp        1.00     -0.86        0.22      0.03
humidity   -0.86      1.00       -0.34     -0.10
wind_speed  0.22     -0.34        1.00      0.13
pressure    0.03     -0.10        0.13      1.00


In [13]:
hot_day = data.loc[data['temp'].idxmax()]
cold_day = data.loc[data['temp'].idxmin()]

print("Hottest day:", hot_day['date'].date(), "->", round(hot_day['temp'], 1), "°C")
print("Coldest day:", cold_day['date'].date(), "->", round(cold_day['temp'], 1), "°C")
print("Average temperature:", round(data['temp'].mean(), 1), "°C")
print("First 7 days average:", round(data['temp'].head(7).mean(), 1), "°C")
print("Last 7 days average:", round(data['temp'].tail(7).mean(), 1), "°C")
print("Average humidity:", round(data['humidity'].mean(), 1))
print("Most common category:", data['temp_category'].mode()[0])

Hottest day: 2017-04-20 -> 34.5 °C
Coldest day: 2017-01-11 -> 11.0 °C
Average temperature: 21.7 °C
First 7 days average: 17.5 °C
Last 7 days average: 33.4 °C
Average humidity: 56.3
Most common category: Mild


In [14]:
data.to_csv("delhi_test_weather_edited.csv", index=False)
files.download("delhi_test_weather_edited.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>